# PPE Compliance Detector — Full Colab Notebook

**ITAI 1378 Midterm Project**

This notebook covers the full pipeline:
1. Setup + GPU check
2. Download dataset from Roboflow
3. Inspect sample images/labels
4. Train YOLOv8n (full run)
5. Evaluate metrics (mAP, precision, recall)
6. (Optional) Train YOLOv8s for comparison
7. Run inference on new images (demo)
8. Save results/weights to Google Drive

**Before running:** Runtime > Change runtime type > Hardware accelerator > GPU (T4)

## 0. Check GPU is active

In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

## 2. (Optional) Mount Google Drive to save results across sessions

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/ITAI1378_PPE_Project"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

## 3. Download dataset from Roboflow

In [ ]:
import roboflow

roboflow.login()

rf = roboflow.Roboflow()
project = rf.workspace("agh-ett2f").project("mask-detection-yolov8")
# confirm the current version number on the Roboflow project page before running
dataset = project.version(16).download("yolov8")
print("Dataset downloaded to:", dataset.location)

## 4. Inspect sample images

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sample_images = glob.glob(f"{dataset.location}/train/images/*.jpg")[:6]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, img_path in zip(axes.flatten(), sample_images):
    ax.imshow(mpimg.imread(img_path))
    ax.axis("off")
plt.tight_layout()
plt.show()

# Check class distribution
!cat {dataset.location}/data.yaml

## 5. Full training run — YOLOv8n baseline

In [ ]:
from ultralytics import YOLO

model_n = YOLO("yolov8n.pt")

results_n = model_n.train(
    data=f"{dataset.location}/data.yaml",
    epochs=75,
    imgsz=640,
    batch=16,
    patience=15,        # early stopping if no improvement
    save_period=10,      # checkpoint every 10 epochs
    project="runs/train",
    name="yolov8n_ppe"
)

## 6. Evaluate YOLOv8n results

In [ ]:
metrics_n = model_n.val()

print("mAP@0.5:", metrics_n.box.map50)
print("mAP@0.5:0.95:", metrics_n.box.map)
print("Precision:", metrics_n.box.mp)
print("Recall:", metrics_n.box.mr)

In [ ]:
# View training curves
from IPython.display import Image, display
display(Image(filename="runs/train/yolov8n_ppe/results.png"))

## 7. (Stretch) Train YOLOv8s for comparison
Only run this if Colab GPU time allows — compare mAP and inference speed against YOLOv8n.

In [ ]:
model_s = YOLO("yolov8s.pt")

results_s = model_s.train(
    data=f"{dataset.location}/data.yaml",
    epochs=75,
    imgsz=640,
    batch=16,
    patience=15,
    save_period=10,
    project="runs/train",
    name="yolov8s_ppe"
)

metrics_s = model_s.val()
print("YOLOv8s mAP@0.5:", metrics_s.box.map50)

## 8. Inference demo — run on a new/test image

In [ ]:
import time

test_image = glob.glob(f"{dataset.location}/test/images/*.jpg")[0]  # swap with your own image path

start = time.time()
results = model_n.predict(source=test_image, conf=0.4, save=True)
elapsed = time.time() - start

print(f"Inference time: {elapsed:.3f} sec")

for r in results:
    im_array = r.plot()  # annotated image as numpy array (BGR)
    plt.figure(figsize=(8, 8))
    plt.imshow(im_array[..., ::-1])  # BGR -> RGB
    plt.axis("off")
    plt.show()

## 9. Simple compliance flag logic
Turns raw detections into a human-readable compliant / non-compliant flag.

In [ ]:
def check_compliance(result, model):
    names = model.names
    detected_classes = [names[int(c)] for c in result.boxes.cls]

    if any("without_mask" in c for c in detected_classes):
        return "\u26a0\ufe0f NON-COMPLIANT: mask missing"
    elif any("incorrectly_worn_mask" in c for c in detected_classes):
        return "\u26a0\ufe0f NON-COMPLIANT: mask worn incorrectly"
    elif any("with_mask" in c for c in detected_classes):
        return "\u2705 COMPLIANT"
    else:
        return "\u2753 No face/mask detected in image"

for r in results:
    print(check_compliance(r, model_n))

## 10. Save best weights to Drive (for demo / submission)

In [ ]:
import shutil

best_weights = "runs/train/yolov8n_ppe/weights/best.pt"
shutil.copy(best_weights, f"{SAVE_DIR}/best_yolov8n_ppe.pt")
print("Saved to", f"{SAVE_DIR}/best_yolov8n_ppe.pt")

## Notes / Troubleshooting
- If Colab disconnects mid-training, checkpoints are saved every 10 epochs (`save_period=10`) inside `runs/train/yolov8n_ppe/weights/`. Resume with `model = YOLO('runs/train/yolov8n_ppe/weights/last.pt')` and call `.train(resume=True)`.
- If GPU is unavailable/queued on Colab free tier, switch to Kaggle Notebooks (free 30 hrs/week GPU) as backup.
- Confirm the Roboflow dataset version number in Section 3 matches the current live version before each run.